<a href="https://colab.research.google.com/github/joanvelastegui/GRC-Listeners/blob/main/RadiosCurrentListeners.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hourly Radio Stream Listener Monitor
This notebook sets up the automatic monitoring for the 14 radio stations. It parses their statistics endpoints, calculates audience analytics, prints the structured report, and persists the historical data into a local CSV file database.

In [5]:
import os
import time
import json
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime
import pytz
import urllib3

# Suppress insecure request warnings from urllib3 when verify=False is used
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuration of the 14 Radio Stations
STATIONS = [
    {"id": 1, "name": "Rumba", "url": "https://radio.kapchosting.com:7052/status-json.xsl", "type": "icecast_json"},
    {"id": 2, "name": "88-5", "url": "https://radio.kapchosting.com:7055/status-json.xsl", "type": "icecast_json"},
    {"id": 3, "name": "Continental", "url": "https://radio.kapchosting.com:7056/status-json.xsl", "type": "icecast_json"},
    {"id": 4, "name": "Amor", "url": "https://rr5100.globalhost1.com:8168/stats", "type": "shoutcast_xml"},
    {"id": 5, "name": "Alegria", "url": "https://rr5100.globalhost1.com:8156/stats", "type": "shoutcast_xml"},
    {"id": 6, "name": "Caracol", "url": "https://rr5100.globalhost1.com:8122/stats", "type": "shoutcast_xml"},
    {"id": 7, "name": "Xtrema", "url": "https://radiostreaming.pro:7213/stats", "type": "shoutcast_xml"},
    {"id": 8, "name": "Centro", "url": "https://dattavolt.com/8080/stats", "type": "shoutcast_xml"},
    {"id": 9, "name": "Ambato", "url": "https://rr5100.globalhost1.com:8140/stats", "type": "shoutcast_xml"},
    {"id": 10, "name": "Retumba", "url": "https://panel.innovatestream.pe:10839/stats", "type": "shoutcast_xml"},
    {"id": 11, "name": "Turbo", "url": "https://streamingecuador.com/stream/radioturboambato", "type": "metadata_only"},
    {"id": 12, "name": "Bandida", "url": "https://sonicpanel.zonaradio.net:8294/stats", "type": "shoutcast_xml"},
    {"id": 13, "name": "Paz y Bien", "url": "https://s7.myradiostream.com:59906/stats", "type": "shoutcast_xml"},
    {"id": 14, "name": "Celeste", "url": "https://streamingecuador.com:9024/status-json.xsl", "type": "icecast_json"}
]

DB_FILE = "radio_listeners_history.csv"
TIMEZONE = pytz.timezone("America/Guayaquil")

def fetch_listener_count(station):
    url = station["url"]
    stype = station["type"]
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) ListenerMonitor/1.0"}

    if stype == "metadata_only":
        return "N/A", "Unavailable", "Direct stream URL lacks stats endpoint"

    # Retry logic
    for attempt in range(3):
        try:
            response = requests.get(url, headers=headers, timeout=12, verify=False)
            if response.status_code == 200:
                content = response.text
                listeners = parse_response(content, stype)
                if listeners is not None:
                    return listeners, "Online", None
                else:
                    return "N/A", "Unavailable", "Could not parse listener element"
            else:
                time.sleep(1)
        except Exception as e:
            if attempt == 2:
                return "N/A", "Unavailable", str(e)
            time.sleep(1)

    return "N/A", "Unavailable", "Connection failed"

def parse_response(content, stype):
    try:
        if stype == "icecast_json":
            data = json.loads(content.strip())
            # Navigate common Icecast JSON structures
            if "icestats" in data:
                stats = data["icestats"]
                if "source" in stats:
                    sources = stats["source"]
                    if isinstance(sources, dict):
                        return int(sources.get("listeners", 0))
                    elif isinstance(sources, list):
                        # Aggregate total listeners across all active mounts
                        return sum(int(src.get("listeners", 0)) for src in sources if "listeners" in src)
        elif stype == "shoutcast_xml":
            root = ET.fromstring(content.strip())
            for tag in ['CURRENTLISTENERS', 'currentlisteners', 'listeners', 'LISTENERS']:
                elem = root.find(tag)
                if elem is not None:
                    return int(elem.text)
    except Exception as e:
        pass
    return None

# Run monitoring cycle
now_local = datetime.now(TIMEZONE)
date_str = now_local.strftime("%Y-%m-%d")
time_str = now_local.strftime("%H:%M")
timestamp_str = now_local.strftime("%Y-%m-%d %H:%M:%S")

results = []
for station in STATIONS:
    listeners, status, err = fetch_listener_count(station)
    results.append({
        "Timestamp": timestamp_str,
        "Radio Station": station["name"],
        "Current Listeners": listeners,
        "Stream Status": status,
        "Collection Error": err if err else ""
    })

# Save to Database
df_current = pd.DataFrame(results)
if not os.path.exists(DB_FILE):
    df_current.to_csv(DB_FILE, index=False)
else:
    df_current.to_csv(DB_FILE, mode='a', header=False, index=False)

# Metrics Calculations
valid_counts = [r["Current Listeners"] for r in results if isinstance(r["Current Listeners"], int)]
total_listeners = sum(valid_counts)
avg_listeners = round(sum(valid_counts) / len(valid_counts), 2) if valid_counts else 0
online_count = sum(1 for r in results if r["Stream Status"] == "Online")
unavailable_count = sum(1 for r in results if r["Stream Status"] == "Unavailable")

# Sort results for presentation: active descending, N/A at the bottom
sorted_results = sorted(
    results,
    key=lambda x: (isinstance(x["Current Listeners"], int), x["Current Listeners"] if isinstance(x["Current Listeners"], int) else -1),
    reverse=True
)

# Station with High/Low
online_stations = [r for r in results if isinstance(r["Current Listeners"], int)]
highest_station = max(online_stations, key=lambda x: x["Current Listeners"])["Radio Station"] if online_stations else "N/A"
highest_listeners = max(online_stations, key=lambda x: x["Current Listeners"])["Current Listeners"] if online_stations else "N/A"
lowest_station = min(online_stations, key=lambda x: x["Current Listeners"])["Radio Station"] if online_stations else "N/A"
lowest_listeners = min(online_stations, key=lambda x: x["Current Listeners"])["Current Listeners"] if online_stations else "N/A"

# Presentation of Report (using sorted_results)
print("HOURLY RADIO LISTENER REPORT (Sorted by Audience)\n")
print(f"Date: {date_str}")
print(f"Time: {time_str}")
print(f"Time Zone: America/Guayaquil\n")
print(f"{'Radio Station':<20} {'Current Listeners':<20} {'Status':<15} {'Contribution':<15}")
print("-" * 75)
for r in sorted_results:
    contrib = "N/A"
    if isinstance(r["Current Listeners"], int) and total_listeners > 0:
        contrib = f"{(r['Current Listeners'] / total_listeners) * 100:.2f}%"
    print(f"{r['Radio Station']:<20} {str(r['Current Listeners']):<20} {r['Stream Status']:<15} {contrib:<15}")
print("-" * 75)
print(f"{'TOTAL':<20} {total_listeners:<20}\n")

# Display calculations and metrics
print("METRICS SUMMARY")
print(f"- Average listeners (active stations): {avg_listeners}")
print(f"- Highest Listener Station: {highest_station} ({highest_listeners})")
print(f"- Lowest Listener Station: {lowest_station} ({lowest_listeners})")
print(f"- Stations Online: {online_count} / Stations Unavailable: {unavailable_count}")

HOURLY RADIO LISTENER REPORT (Sorted by Audience)

Date: 2026-11-06
Time: 11:13
Time Zone: America/Guayaquil

Radio Station        Current Listeners    Status          Contribution   
---------------------------------------------------------------------------
Alegria              866                  Online          37.52%         
Caracol              639                  Online          27.69%         
Celeste              320                  Online          13.86%         
Rumba                194                  Online          8.41%          
Amor                 71                   Online          3.08%          
Centro               56                   Online          2.43%          
Xtrema               42                   Online          1.82%          
88-5                 38                   Online          1.65%          
Ambato               29                   Online          1.26%          
Paz y Bien           28                   Online          1.21%          
